#Set up

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# Função do gráfico

In [ ]:
def plot_barra_periodo(
    categorias,
    valores,
    ultimos_n=6,
    titulo="Variações do Indicador ao longo do Período",
    lag_yoy=12,
    incluir_pop=True,
    incluir_yoy=True,
    incluir_media=True,
    destacar_max_min=True,
    evitar_destacar_max_min_adjacentes=True,
    cor_max="#2E8B57",
    cor_min="#C95A5A",
    cor_acima_media="#4E79A7",
    cor_abaixo_media="#A0CBE8",
    cor_media="#B5BD2B",
    cor_pop_positivo="#2CA02C",
    cor_pop_negativo="#D62728",
    cor_yoy_positivo="#2CA02C",
    cor_yoy_negativo="#D62728",
    largura_px=1000,
    altura_px=620,
    margem_esquerda=40,
    margem_direita=140,
    margem_superior_com_legenda=150,
    margem_superior_sem_legenda=110,
    margem_inferior=60,
    bargap=0.18,
    tamanho_fonte_titulo=13,
    tamanho_fonte_eixo_x=13,
    tamanho_fonte_eixo_y=13,
    tamanho_fonte_titulo_eixo_x=12,
    tamanho_fonte_titulo_eixo_y=12,
    tamanho_fonte_valor_barra=13,
    tamanho_fonte_variacao=11,
    tamanho_fonte_media=13,
    tamanho_fonte_legenda=11,
    cor_fonte_valor_barra="white",
    line_width_media=2,
    line_dash_media="dash",
    offset_pop_relativo=0.08,
    offset_yoy_relativo=0.16,
    multiplicador_limite_y=1.38,
    mostrar_legenda=False,
    mostrar_hover=True,
    texto_hover="<b>%{x}</b><br>Valor: %{y:.1f}<extra></extra>",
    titulo_eixo_x="",
    titulo_eixo_y="",
    standoff_eixo_x=10,
    standoff_eixo_y=10,
    mostrar_eixo_y=False,
    mostrar_grid_y=False,
    cor_grid_y="#E5E5E5",
    formato_tick_y=None,
):
    """
    Cria um gráfico de barras analítico com possibilidade de exibir:
    - cores acima e abaixo da média
    - destaque para valor máximo e mínimo
    - linha da média
    - PoP
    - YoY
    - legenda opcional
    - títulos de eixo
    - controle do eixo Y e grid

    Toda a lógica auxiliar fica dentro da própria função principal,
    para facilitar uso didático em notebooks e aulas.

    Parâmetros novos relacionados aos eixos:
    - titulo_eixo_x: título do eixo X
    - titulo_eixo_y: título do eixo Y
    - tamanho_fonte_titulo_eixo_x: tamanho da fonte do título do eixo X
    - tamanho_fonte_titulo_eixo_y: tamanho da fonte do título do eixo Y
    - standoff_eixo_x: distância entre o título do eixo X e o eixo
    - standoff_eixo_y: distância entre o título do eixo Y e o eixo
    - mostrar_eixo_y: se True, exibe o eixo Y
    - mostrar_grid_y: se True, mostra linhas de grade horizontais
    - cor_grid_y: cor da grade do eixo Y
    - formato_tick_y: formato dos ticks do eixo Y
      Exemplos:
      - ",.0f"  -> separador de milhar
      - ".1f"   -> uma casa decimal
      - ".1%"   -> percentual
    """

    import numpy as np
    import plotly.graph_objects as go

    # ==========================================================
    # 1. PADRONIZAÇÃO E VALIDAÇÕES INICIAIS
    # ==========================================================
    # Garante que 'categorias' seja uma lista e 'valores' um array NumPy para consistência e operações eficientes.
    categorias = list(categorias)
    valores = np.array(valores, dtype=float)

    # Validação crucial: verifica se o número de categorias corresponde ao número de valores.
    # Se não, levanta um erro, pois os dados seriam inconsistentes para plotagem.
    if len(categorias) != len(valores):
        raise ValueError("categorias e valores precisam ter o mesmo tamanho.")

    # Validação para cálculo de YoY (Year-over-Year).
    # Se a inclusão de YoY for solicitada, verifica se há dados suficientes para o lag definido.
    if incluir_yoy and len(valores) < lag_yoy + 1:
        raise ValueError(
            f"Para incluir YoY com lag_yoy={lag_yoy}, é preciso ter pelo menos {lag_yoy + 1} valores."
        )

    # ==========================================================
    # 2. FUNÇÕES AUXILIARES INTERNAS
    #    Estas funções são definidas localmente para encapsular lógica específica
    #    e manter o escopo da função principal mais limpo.
    # ==========================================================
    def _recortar_serie(categorias_base, valores_base, n):
        """
        Recorta as séries de categorias e valores para exibir apenas os 'n' últimos períodos.
        Útil para focar em dados mais recentes no gráfico.
        """
        if n is None: # Se 'n' for None, retorna a série completa sem recorte.
            return categorias_base, valores_base
        return categorias_base[-n:], valores_base[-n:] # Retorna os últimos 'n' elementos.

    def _calcular_pop(valores_base):
        """
        Calcula a variação PoP (Period-over-Period) para cada valor na série.
        Retorna uma lista de tuplas (delta, percentual de variação) ou None para o primeiro elemento.
        """
        resultado = [None] # O primeiro período não tem PoP, então é None.
        for i in range(1, len(valores_base)): # Começa do segundo elemento para comparar com o anterior.
            valor_anterior = valores_base[i - 1]
            delta = valores_base[i] - valor_anterior # Diferença absoluta.
            # Calcula o percentual de variação. Evita divisão por zero.
            perc = (delta / valor_anterior) * 100 if valor_anterior != 0 else 0
            resultado.append((delta, perc))
        return resultado

    def _calcular_yoy(valores_base, lag):
        """
        Calcula a variação YoY (Year-over-Year) para cada valor, baseado em um 'lag' específico.
        Geralmente, 'lag' é 12 para comparações anuais em dados mensais.
        """
        resultado = [None] * lag # Os primeiros 'lag' períodos não têm YoY.
        for i in range(lag, len(valores_base)): # Começa a partir do índice 'lag' para ter referência.
            valor_referencia = valores_base[i - lag]
            delta = valores_base[i] - valor_referencia # Diferença absoluta.
            # Calcula o percentual de variação. Evita divisão por zero.
            perc = (delta / valor_referencia) * 100 if valor_referencia != 0 else 0
            resultado.append((delta, perc))
        return resultado

    def _definir_cores_barras(valores_base, media_base):
        """
        Define as cores para cada barra com base em seu valor em relação à média
        e nos destaques de máximo/mínimo, se ativados.
        """
        vmax_local = valores_base.max() # Encontra o valor máximo na série atual.
        vmin_local = valores_base.min() # Encontra o valor mínimo na série atual.

        # Encontra os índices dos valores máximo e mínimo.
        idx_max = np.where(valores_base == vmax_local)[0]
        idx_min = np.where(valores_base == vmin_local)[0]

        # Verifica se os valores máximo e mínimo são adjacentes, o que pode causar sobreposição de destaque.
        adjacentes = any(abs(i - j) == 1 for i in idx_max for j in idx_min)

        usar_destaque = destacar_max_min # Inicia com a configuração do parâmetro.
        # Se evitar destaque adjacente for True E houver adjacência, desativa o destaque.
        if evitar_destacar_max_min_adjacentes and adjacentes:
            usar_destaque = False

        cores_locais = []
        for i, v in enumerate(valores_base): # Itera sobre cada valor para atribuir uma cor.
            if usar_destaque and i in idx_max: # Se destaque ativado e for o valor máximo.
                cores_locais.append(cor_max)
            elif usar_destaque and i in idx_min: # Se destaque ativado e for o valor mínimo.
                cores_locais.append(cor_min)
            elif v >= media_base: # Se o valor estiver acima ou igual à média.
                cores_locais.append(cor_acima_media)
            else: # Se o valor estiver abaixo da média.
                cores_locais.append(cor_abaixo_media)

        return cores_locais, usar_destaque # Retorna as cores e se o destaque foi efetivamente usado.

    def _formatar_texto_variacao(tipo, delta, perc):
        """
        Formata o texto para as anotações de PoP ou YoY (delta e percentual).
        Inclui setas indicativas de aumento/diminuição.
        """
        if delta > 0:
            seta = "▲" # Seta para cima para aumento.
            sinal = "+"
        elif delta < 0:
            seta = "▼" # Seta para baixo para diminuição.
            sinal = ""
        else:
            seta = "•" # Ponto para variação zero.
            sinal = ""

        delta_txt = f"{sinal}{int(round(delta))}" # Formata o delta (valor absoluto).
        perc_txt = f"{sinal}{perc:.1f}%" # Formata o percentual com uma casa decimal.
        return f"{tipo}: {seta} {delta_txt} ({perc_txt})"

    def _cor_variacao(delta, cor_positiva, cor_negativa):
        """
        Retorna a cor apropriada para a anotação de variação (PoP/YoY) com base no delta.
        """
        return cor_positiva if delta > 0 else cor_negativa

    def _adicionar_legenda_manual(fig_local, usar_destaque_local):
        """
        Adiciona entradas de legenda 'fantasmas' ao gráfico. Isso é feito com traces invisíveis
        para ter controle total sobre o texto e a aparência da legenda, sem exibir traces de dados.
        """
        # Legenda para destaque de máximo/mínimo (se ativado)
        if usar_destaque_local:
            fig_local.add_trace(
                go.Bar(
                    x=[None], y=[None], # Traces com dados None são invisíveis, apenas para legenda.
                    marker_color=cor_max,
                    name="Valor máximo",
                    showlegend=True, # Mostrar na legenda.
                    hoverinfo="skip", # Não mostrar hover para este trace.
                )
            )
            fig_local.add_trace(
                go.Bar(
                    x=[None], y=[None],
                    marker_color=cor_min,
                    name="Valor mínimo",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )

        # Legenda para barras acima e abaixo da média
        fig_local.add_trace(
            go.Bar(
                x=[None], y=[None],
                marker_color=cor_acima_media,
                name="Acima da média",
                showlegend=True,
                hoverinfo="skip",
            )
        )
        fig_local.add_trace(
            go.Bar(
                x=[None], y=[None],
                marker_color=cor_abaixo_media,
                name="Abaixo da média",
                showlegend=True,
                hoverinfo="skip",
            )
        )

        # Legenda para a linha da média (se incluída)
        if incluir_media:
            fig_local.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode="lines", # Tipo de trace para linha.
                    line=dict(color=cor_media, dash=line_dash_media, width=line_width_media),
                    name="Média",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )

        # Legenda para PoP (se incluído) - separado por positivo/negativo.
        if incluir_pop:
            fig_local.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode="markers", # Tipo de trace para marcador.
                    marker=dict(color=cor_pop_positivo, size=10, symbol="triangle-up"),
                    name="PoP, aumento",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )
            fig_local.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode="markers",
                    marker=dict(color=cor_pop_negativo, size=10, symbol="triangle-down"),
                    name="PoP, diminuição",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )

        # Legenda para YoY (se incluído) - separado por positivo/negativo.
        if incluir_yoy:
            fig_local.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode="markers",
                    marker=dict(color=cor_yoy_positivo, size=10, symbol="triangle-up"),
                    name="YoY, aumento",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )
            fig_local.add_trace(
                go.Scatter(
                    x=[None], y=[None],
                    mode="markers",
                    marker=dict(color=cor_yoy_negativo, size=10, symbol="triangle-down"),
                    name="YoY, diminuição",
                    showlegend=True,
                    hoverinfo="skip",
                )
            )

    # ==========================================================
    # 3. RECORTE DA JANELA EXIBIDA
    # ==========================================================
    # Aplica a função auxiliar para obter apenas os 'n' últimos períodos, se especificado.
    categorias_plot, valores_plot = _recortar_serie(categorias, valores, ultimos_n)

    # ==========================================================
    # 4. MÉTRICAS PRINCIPAIS
    # ==========================================================
    media = valores_plot.mean() # Calcula a média dos valores a serem plotados.
    vmax = valores_plot.max() # Encontra o valor máximo dos valores a serem plotados.

    # ==========================================================
    # 5. CÁLCULO DE CORES
    # ==========================================================
    # Determina as cores de cada barra e se o destaque de max/min foi realmente aplicado.
    cores_barras, usar_destaque_efetivo = _definir_cores_barras(valores_plot, media)

    # ==========================================================
    # 6. CÁLCULO DE POP E YOY
    # ==========================================================
    # Calcula PoP e YoY para toda a série original, pois o YoY pode precisar de dados anteriores aos 'ultimos_n'.
    pop_completo = _calcular_pop(valores)
    yoy_completo = _calcular_yoy(valores, lag_yoy)

    # Recorta os resultados de PoP e YoY para corresponder aos 'ultimos_n' períodos a serem plotados.
    if ultimos_n is None:
        pop_plot = pop_completo
        yoy_plot = yoy_completo
    else:
        pop_plot = pop_completo[-ultimos_n:]
        yoy_plot = yoy_completo[-ultimos_n:]

    # ==========================================================
    # 7. AJUSTES DE TÍTULO, LEGENDA E MARGEM SUPERIOR
    #    Define posições e margens com base na decisão de mostrar a legenda.
    # ==========================================================
    if mostrar_legenda:
        titulo_y = 0.90 # Posição Y do título quando há legenda.
        legenda_y = 1.09 # Posição Y da legenda (acima do gráfico).
        margem_superior = margem_superior_com_legenda # Usa a margem superior maior para acomodar a legenda.
    else:
        titulo_y = 0.90 # Posição Y do título quando não há legenda.
        legenda_y = None # Nenhuma legenda, então sem posição Y.
        margem_superior = margem_superior_sem_legenda # Usa a margem superior menor.

    # ==========================================================
    # 8. CRIAÇÃO DA FIGURA
    # ==========================================================
    fig = go.Figure() # Inicializa o objeto figura do Plotly.

    # ==========================================================
    # 9. CAMADA PRINCIPAL DE BARRAS
    # ==========================================================
    fig.add_trace(
        go.Bar(
            x=categorias_plot, # Categorias no eixo X.
            y=valores_plot, # Valores no eixo Y.
            marker_color=cores_barras, # Cores definidas anteriormente para cada barra.
            text=[str(int(round(v))) for v in valores_plot], # Texto exibido dentro/sobre as barras.
            textposition="inside", # Posição do texto dentro da barra.
            textfont=dict(color=cor_fonte_valor_barra, size=tamanho_fonte_valor_barra),
            insidetextanchor="end", # Ancoragem do texto dentro da barra.
            showlegend=False, # Este trace não deve aparecer na legenda automática.
            hovertemplate=texto_hover if mostrar_hover else None, # Template de hover, se ativado.
            hoverinfo="skip" if not mostrar_hover else None, # Pula hoverinfo se não mostrar hover.
        )
    )

    # ==========================================================
    # 10. ANOTAÇÕES DE POP (Period-over-Period)
    # ==========================================================
    if incluir_pop:
        # Calcula o offset vertical para as anotações PoP, baseado no valor máximo.
        offset_pop = vmax * offset_pop_relativo

        # Itera sobre os dados a serem plotados e seus respectivos cálculos PoP.
        for x, v, p in zip(categorias_plot, valores_plot, pop_plot):
            if p is None: # Pula se não houver cálculo PoP (primeiro período).
                continue

            delta, perc = p # Desempacota o delta e o percentual de variação.
            texto_pop = _formatar_texto_variacao("PoP", delta, perc) # Formata o texto da anotação.
            cor_pop = _cor_variacao(delta, cor_pop_positivo, cor_pop_negativo) # Define a cor com base no delta.

            fig.add_annotation(
                x=x,
                y=v + offset_pop, # Posição Y acima da barra, com offset.
                text=texto_pop,
                showarrow=False, # Não exibe a seta da anotação.
                font=dict(color=cor_pop, size=tamanho_fonte_variacao),
                xanchor="center", # Ancoragem horizontal do texto.
                yanchor="bottom", # Ancoragem vertical do texto.
                align="center", # Alinhamento do texto.
            )

    # ==========================================================
    # 11. ANOTAÇÕES DE YOY (Year-over-Year)
    # ==========================================================
    if incluir_yoy:
        # Calcula o offset vertical para as anotações YoY, ligeiramente acima do PoP.
        offset_yoy = vmax * offset_yoy_relativo

        # Itera sobre os dados a serem plotados e seus respectivos cálculos YoY.
        for x, v, y_item in zip(categorias_plot, valores_plot, yoy_plot):
            if y_item is None: # Pula se não houver cálculo YoY (períodos iniciais).
                continue

            delta, perc = y_item # Desempacota o delta e o percentual de variação.
            texto_yoy = _formatar_texto_variacao("YoY", delta, perc) # Formata o texto da anotação.
            cor_yoy = _cor_variacao(delta, cor_yoy_positivo, cor_yoy_negativo) # Define a cor com base no delta.

            fig.add_annotation(
                x=x,
                y=v + offset_yoy, # Posição Y acima da barra e da anotação PoP.
                text=texto_yoy,
                showarrow=False,
                font=dict(color=cor_yoy, size=tamanho_fonte_variacao),
                xanchor="center",
                yanchor="bottom",
                align="center",
            )

    # ==========================================================
    # 12. LINHA E RÓTULO DA MÉDIA
    # ==========================================================
    if incluir_media:
        fig.add_hline(
            y=media, # Desenha uma linha horizontal na posição da média.
            line_dash=line_dash_media,
            line_color=cor_media,
            line_width=line_width_media,
        )

        fig.add_annotation(
            x=1.01, # Posição X da anotação (fora da área do gráfico, à direita).
            xref="paper", # Refere-se à largura total do 'papel' do gráfico.
            y=media, # Posição Y na altura da linha da média.
            yref="y", # Refere-se ao sistema de coordenadas do eixo Y.
            text=f"Média: {int(round(media))}", # Texto da anotação (valor da média).
            showarrow=False,
            xanchor="left", # Ancoragem à esquerda.
            yanchor="middle", # Ancoragem ao meio.
            font=dict(color=cor_media, size=tamanho_fonte_media),
        )

    # ==========================================================
    # 13. LEGENDA MANUAL
    # ==========================================================
    if mostrar_legenda:
        _adicionar_legenda_manual(fig, usar_destaque_efetivo) # Chama a função auxiliar para criar a legenda.

    # ==========================================================
    # 14. LAYOUT
    # ==========================================================
    # Dicionário para armazenar os argumentos de layout, permitindo flexibilidade.
    layout_kwargs = dict(
        title=dict(
            text=titulo, # Título principal do gráfico.
            x=0.5, # Centraliza o título horizontalmente.
            xanchor="center",
            y=titulo_y, # Posição Y do título (ajustada para legenda).
            font=dict(size=tamanho_fonte_titulo),
        ),
        width=largura_px, # Largura da figura.
        height=altura_px, # Altura da figura.
        plot_bgcolor="white", # Cor de fundo da área de plotagem.
        paper_bgcolor="white", # Cor de fundo do 'papel' da figura.
        margin=dict( # Margens ao redor do gráfico.
            l=margem_esquerda,
            r=margem_direita,
            t=margem_superior,
            b=margem_inferior,
        ),
        bargap=bargap, # Espaçamento entre as barras.
        showlegend=mostrar_legenda, # Controla a exibição da legenda geral (mas usamos traces fantasmas).
    )

    # Adiciona configurações específicas da legenda se ela for mostrada.
    if mostrar_legenda:
        layout_kwargs["legend"] = dict(
            orientation="h", # Orientação horizontal da legenda.
            x=0.5, # Centraliza a legenda horizontalmente.
            xanchor="center",
            y=legenda_y, # Posição Y da legenda.
            font=dict(size=tamanho_fonte_legenda),
        )

    fig.update_layout(**layout_kwargs) # Aplica todas as configurações de layout à figura.

    # ==========================================================
    # 15. EIXO X
    # ==========================================================
    fig.update_xaxes(
        showline=True, # Exibe a linha do eixo X.
        linecolor="#BFBFBF", # Cor da linha do eixo X.
        tickfont=dict(size=tamanho_fonte_eixo_x), # Tamanho da fonte dos ticks do eixo X.
        title=dict(
            text=titulo_eixo_x, # Título do eixo X.
            font=dict(size=tamanho_fonte_titulo_eixo_x),
            standoff=standoff_eixo_x, # Distância entre o título e o eixo.
        ),
    )

    # ==========================================================
    # 16. EIXO Y
    # ==========================================================
    fig.update_yaxes(
        visible=mostrar_eixo_y, # Controla a visibilidade do eixo Y.
        range=[0, vmax * multiplicador_limite_y], # Define o range do eixo Y, com um multiplicador para "espaço extra" no topo.
        tickfont=dict(size=tamanho_fonte_eixo_y), # Tamanho da fonte dos ticks do eixo Y.
        title=dict(
            text=titulo_eixo_y, # Título do eixo Y.
            font=dict(size=tamanho_fonte_titulo_eixo_y),
            standoff=standoff_eixo_y, # Distância entre o título e o eixo.
        ),
        tickformat=formato_tick_y, # Formato dos rótulos dos ticks do eixo Y (ex: percentual, moeda).
        showgrid=mostrar_grid_y, # Exibe a grade horizontal do eixo Y.
        gridcolor=cor_grid_y, # Cor da grade do eixo Y.
        zeroline=False, # Não exibe a linha do zero no eixo Y.
    )

    return fig # Retorna a figura Plotly configurada.

# Dados de demostração

In [ ]:
def gerar_dados_fake_barras(
    n_periodos=24,
    seed=42,
    data_inicial="2023-01-01",
    frequencia="ME",
    formato_categoria="%b %Y",
):
    """
    Gera dados sintéticos para testar gráficos de barras temporais.

    Parâmetros
    ----------
    n_periodos : int
        Quantidade de períodos a serem gerados.
    seed : int
        Semente para reprodutibilidade.
    data_inicial : str
        Data inicial da série.
    frequencia : str
        Frequência da série temporal no padrão do pandas.
    formato_categoria : str
        Formato textual das datas no eixo X.

    Retorno
    -------
    tuple
        categorias : list[str]
        valores : np.ndarray
    """
    np.random.seed(seed)

    base = 100
    tendencia = np.linspace(0, 20, n_periodos)
    sazonalidade = 8 * np.sin(np.linspace(0, 3 * np.pi, n_periodos))
    ruido = np.random.normal(0, 6, n_periodos)

    valores = base + tendencia + sazonalidade + ruido
    valores = np.round(valores, 1)

    datas = pd.date_range(start=data_inicial, periods=n_periodos, freq=frequencia)
    categorias = [d.strftime(formato_categoria) for d in datas]

    return categorias, valores


# Plotagem

In [ ]:
# ==============================================================
# EXEMPLO DE USO
# ==============================================================
categorias, valores = gerar_dados_fake_barras()

fig = plot_barra_periodo(
    categorias=categorias,
    valores=valores,
    ultimos_n=6,
    titulo="Variações do Indicador ao longo do Período",
    lag_yoy=12,
    incluir_pop=True,
    incluir_yoy=True,
    incluir_media=True,
    destacar_max_min=True,
    evitar_destacar_max_min_adjacentes=True,
    mostrar_legenda=True,
    largura_px=1000,
    altura_px=620,
)

fig.show()